# « L'inflation ralentit » ne veut pas dire « les prix baissent » · *« Inflation is slowing » does not mean « prices are falling »*

Notebook compagnon du chapitre **5. Le vocabulaire essentiel de la macroéconomie financière, sans jargon** — [lire l'article](https://nmlab.io/ressources/le-vocabulaire-essentiel-de-la-macroeconomie-financiere).
Companion notebook to chapter **5. The Essential Vocabulary of Financial Macroeconomics, Without the Jargon** — [read the article](https://nmlab.io/en/ressources/the-essential-vocabulary-of-financial-macroeconomics).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_prices() -> tuple[Series, Series]:
    """Indice des prix (CPIAUCSL) depuis 2018 et son glissement annuel — l'inflation — depuis 2019.
    Live from FRED: CPI level since 2018 and its year-over-year change (inflation) since 2019."""
    cpi = nm.load_fred("CPIAUCSL").loc["2018":]
    inflation = ((cpi / cpi.shift(12) - 1) * 100).loc["2019":]
    return cpi, inflation

cpi, inflation = load_prices()
print(f"Dernier point / latest: {inflation.index[-1]:%Y-%m} → {inflation.iloc[-1]:.1f} %")


import pandas as pd
import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="« L'inflation ralentit » ne veut pas dire « les prix baissent »",
        sub="États-Unis, 2018-2025 — le même indice des prix, lu de deux façons",
        lvl="Le niveau : l'indice des prix (IPC)", chg="La variation : l'inflation sur un an",
        lvl_ann="les prix continuent\nde monter…", chg_ann="…pendant que\nl'inflation retombe :\nla désinflation",
        peak="pic : ≈ 9 %", target="cible : 2 %",
        note="Source : Bureau of Labor Statistics, via FRED (série CPIAUCSL). Zone grisée : la désinflation de 2022-2025 —\n"
             "la courbe de droite descend, celle de gauche monte toujours."),
    "en": dict(
        title="« Inflation is slowing » does not mean « prices are falling »",
        sub="United States, 2018-2025 — the same price index, read two ways",
        lvl="The level: the price index (CPI)", chg="The change: inflation year over year",
        lvl_ann="prices keep\nrising…", chg_ann="…while inflation\nfalls back:\ndisinflation",
        peak="peak: ≈ 9%", target="target: 2%",
        note="Source: Bureau of Labor Statistics, via FRED (CPIAUCSL series). Shaded zone: the 2022-2025 disinflation —\n"
             "the right curve falls, the left one still rises."),
}

def build_figure(cpi: Series, inflation: Series, lang: str) -> Figure:
    """Deux panneaux : le niveau de l'IPC (gauche) et sa variation annuelle (droite)."""
    text = LABELS[lang]
    fig = nm.figure(height_px=874)
    left = fig.add_axes([0.058, 0.165, 0.395, 0.515])
    right = fig.add_axes([0.567, 0.165, 0.415, 0.515])
    band0 = pd.Timestamp("2022-07-01")

    left.axvspan(band0, cpi.index[-1], color=nm.COLORS["edge"], alpha=0.4, linewidth=0)
    left.plot(cpi.index, cpi, color=nm.COLORS["blue"], linewidth=3)
    left.set_ylim(240, 340)
    left.set_yticks(range(240, 341, 20))
    left.set_title(text["lvl"], loc="left", color=nm.COLORS["text"], fontsize=23, fontweight="bold", pad=12)
    left.text(0.62, 0.26, text["lvl_ann"], transform=left.transAxes, fontsize=21,
              fontweight="bold", color=nm.COLORS["blue"], linespacing=1.4)

    right.axvspan(band0, inflation.index[-1], color=nm.COLORS["edge"], alpha=0.4, linewidth=0)
    right.axhline(2, color=nm.COLORS["muted"], linestyle=(0, (6, 4)), linewidth=2, alpha=0.9)
    right.plot(inflation.index, inflation, color=nm.COLORS["rose"], linewidth=3)
    peak = inflation.idxmax()
    right.scatter([peak], [inflation.max()], s=70, color=nm.COLORS["rose"], zorder=5)
    right.set_ylim(0, 10.5)
    right.set_yticks(range(0, 11, 2))
    right.set_yticklabels([f"{v} %" for v in range(0, 11, 2)])
    right.set_title(text["chg"], loc="left", color=nm.COLORS["text"], fontsize=23, fontweight="bold", pad=12)
    right.annotate(text["peak"], xy=(peak, inflation.max()), xytext=(14, 2),
                   textcoords="offset points", ha="left", va="center", fontsize=20, color=nm.COLORS["text"])
    right.text(0.635, 0.72, text["chg_ann"], transform=right.transAxes, va="top", fontsize=21,
               fontweight="bold", color=nm.COLORS["rose"], linespacing=1.45)
    right.text(0.975, 0.15, text["target"], transform=right.transAxes, ha="right", va="center",
               fontsize=20, color=nm.COLORS["muted"])

    for ax, series in ((left, cpi), (right, inflation)):
        ax.set_xlim(series.index[0], series.index[-1])
        ax.margins(x=0.01)
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(cpi, inflation, LANG)